# 06 — Ablation Study
**BraTS 2023 GLI Brain Tumor Segmentation | RCOEM 2026–27**

This notebook compares the 3 ablation variants:
1. **Baseline** — plain 3D U-Net (no attention)
2. **Channel Attn only** — SE blocks, no spatial gates
3. **Spatial Attn only** — spatial attention gates, no SE
4. **Full (Proposed)** — both SE + spatial attention

Presents statistical analysis showing each component's contribution.

**Estimated time:** ~2 hours (if retraining ablation variants), or ~10 min if using saved checkpoints

In [ ]:
import sys, os
DATASET_PATH = '/home/yourname/BraTS2023_Training_Data'  # ← CHANGE THIS
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import torch
from tqdm.notebook import tqdm

from src.models  import UNet3D, AttentionUNet3D
from src.dataset import get_patient_folders, load_patient, get_file_paths
from src.metrics import compute_patient_metrics, aggregate_metrics, print_metrics_table
from src.utils   import load_checkpoint, sliding_window_inference
from src.config  import DEVICE, PATCH_SIZE, PATCH_OVERLAP, MAX_PATIENTS, RANDOM_SEED

print(f'Device: {DEVICE}')
print('✅ Ablation study setup ready')

## 1️⃣ Define Ablation Variants

In [ ]:
ABLATION_CONFIGS = [
    {
        'name':        'Baseline (No Attention)',
        'model_name':  'baseline_unet3d',
        'model':       UNet3D(4, 4, 32),
        'ch_attn':     False,
        'sp_attn':     False,
    },
    {
        'name':        'Channel Attention Only (SE)',
        'model_name':  'attention_unet3d_channel_only',
        'model':       AttentionUNet3D(4, 4, 32, use_channel_attn=True, use_spatial_attn=False),
        'ch_attn':     True,
        'sp_attn':     False,
    },
    {
        'name':        'Spatial Attention Only (Gates)',
        'model_name':  'attention_unet3d_spatial_only',
        'model':       AttentionUNet3D(4, 4, 32, use_channel_attn=False, use_spatial_attn=True),
        'ch_attn':     False,
        'sp_attn':     True,
    },
    {
        'name':        'Full Attention — Proposed',
        'model_name':  'attention_unet3d_full',
        'model':       AttentionUNet3D(4, 4, 32, use_channel_attn=True, use_spatial_attn=True),
        'ch_attn':     True,
        'sp_attn':     True,
    },
]

for cfg in ABLATION_CONFIGS:
    cfg['model'] = cfg['model'].to(DEVICE)
    _, best_dice = load_checkpoint(cfg['model'], optimizer=None,
                                   model_name=cfg['model_name'], prefer_best=True)
    cfg['model'].eval()
    params = sum(p.numel() for p in cfg['model'].parameters())
    print(f"  {cfg['name']:45s} | Params: {params:,} | Best Val Dice: {best_dice:.4f}")

## 2️⃣ Get Test Set & Run Inference

In [ ]:
import nibabel as nib

np.random.seed(RANDOM_SEED)
all_folders = get_patient_folders(DATASET_PATH)
if MAX_PATIENTS:
    all_folders = all_folders[:MAX_PATIENTS]
np.random.shuffle(all_folders)
n_test = max(1, int(len(all_folders) * 0.10))
test_folders = all_folders[-n_test:]
print(f'Test set: {len(test_folders)} patients')

all_ablation_results = {}

for cfg in ABLATION_CONFIGS:
    print(f"\nRunning: {cfg['name']}")
    patient_results = []
    for folder in tqdm(test_folders, desc=cfg['name'][:30]):
        image, gt_seg = load_patient(folder, has_seg=True)
        paths = get_file_paths(folder)
        nii   = nib.load(paths['flair'])
        voxel_spacing = tuple(abs(nii.header.get_zooms()[:3]))
        image_tensor = torch.from_numpy(image.astype(np.float32))
        with torch.no_grad():
            pred_seg = sliding_window_inference(
                cfg['model'], image_tensor, PATCH_SIZE, PATCH_OVERLAP, DEVICE
            )
        metrics = compute_patient_metrics(pred_seg, gt_seg, voxel_spacing)
        patient_results.append(metrics)
    all_ablation_results[cfg['name']] = patient_results
    agg = aggregate_metrics(patient_results)
    print_metrics_table(agg, cfg['name'])

print('\n✅ All ablation variants evaluated')

## 3️⃣ Ablation Table (Report-Ready)

In [ ]:
rows = []
for cfg in ABLATION_CONFIGS:
    agg = aggregate_metrics(all_ablation_results[cfg['name']])
    rows.append({
        'Model':         cfg['name'],
        'Channel Attn':  '✓' if cfg['ch_attn'] else '✗',
        'Spatial Attn':  '✓' if cfg['sp_attn'] else '✗',
        'Dice WT':   round(agg['WT']['dice']['mean'], 4),
        'Dice TC':   round(agg['TC']['dice']['mean'], 4),
        'Dice ET':   round(agg['ET']['dice']['mean'], 4),
        'HD95 WT↓':  round(agg['WT']['hd95']['mean'], 2),
        'HD95 ET↓':  round(agg['ET']['hd95']['mean'], 2),
    })

ablation_df = pd.DataFrame(rows)
ablation_df.to_csv('results/ablation_study.csv', index=False)
print('📊 Ablation results saved to results/ablation_study.csv')
print('\n📋 Ablation Study Table:')
print(ablation_df.to_string(index=False))

## 4️⃣ Statistical Significance (Wilcoxon Signed-Rank Test)

In [ ]:
print('Statistical Significance: Baseline vs Proposed (Full Attention)')
print('Using Wilcoxon signed-rank test (non-parametric, paired)\n')

baseline_results  = all_ablation_results['Baseline (No Attention)']
proposed_results  = all_ablation_results['Full Attention — Proposed']

for region in ['WT', 'TC', 'ET']:
    baseline_dices = [r[region]['dice'] for r in baseline_results]
    proposed_dices = [r[region]['dice'] for r in proposed_results]

    stat, p_value = stats.wilcoxon(proposed_dices, baseline_dices, alternative='greater')
    sig = '✅ Significant (p<0.05)' if p_value < 0.05 else '❌ Not significant'
    print(f'  {region}: Baseline={np.mean(baseline_dices):.4f} '
          f'| Proposed={np.mean(proposed_dices):.4f} '
          f'| p={p_value:.4f} | {sig}')

## 5️⃣ Ablation Contribution Bar Chart

In [ ]:
model_labels  = ['Baseline', '+ Channel\nAttn (SE)', '+ Spatial\nAttn Gates', 'Full\n(Ours)']
colors_ablation = ['#95a5a6', '#5dade2', '#a9cce3', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ablation Study — Effect of Each Attention Component', fontsize=13, fontweight='bold')

for ax, region, region_color in zip(axes, ['WT', 'TC', 'ET'], ['#3498db','#e74c3c','#2ecc71']):
    dice_scores = []
    for cfg in ABLATION_CONFIGS:
        agg = aggregate_metrics(all_ablation_results[cfg['name']])
        dice_scores.append(agg[region]['dice']['mean'])

    bars = ax.bar(model_labels, dice_scores, color=colors_ablation, edgecolor='white', linewidth=0.5)

    for bar, val in zip(bars, dice_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    baseline_dice = dice_scores[0]
    ax.axhline(y=baseline_dice, color='gray', ls='--', alpha=0.6, label='Baseline')

    ax.set_title(f'Dice — {region}', fontsize=12)
    ax.set_ylabel('Mean Dice Score')
    ax.set_ylim(max(0, min(dice_scores) - 0.05), min(1.0, max(dice_scores) + 0.05))
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/ablation_barchart.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: results/ablation_barchart.png')
print('\n🏁 Notebook 06 complete. Proceed to 07_visualise_results.ipynb')